# Taller de implementación: Generador de tabla predictiva LL(1)

**Objetivo:** construir automáticamente los conjuntos FIRST, FOLLOW y la tabla predictiva LL(1) a partir de una gramática independiente del contexto.

**Gramática de trabajo**

```
E  -> T E'
E' -> + T E' | ε
T  -> F T'
T' -> * F T' | ε
F  -> ( E ) | id
```

Todos los conjuntos y la tabla se calculan con algoritmos implementados aquí (punto fijo iterativo); ninguno se ingresa manualmente.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Taller de implementación: Generador de tabla predictiva LL(1)

Construye automáticamente, a partir de una gramática independiente del
contexto, los conjuntos FIRST, FOLLOW y la tabla de análisis predictivo LL(1).

Gramática de trabajo (expresiones aritméticas):

    E  -> T E'
    E' -> + T E' | ε
    T  -> F T'
    T' -> * F T' | ε
    F  -> ( E ) | id
"""

# ---------------------------------------------------------------------------
# 1. Representación de la gramática mediante un diccionario de producciones
# ---------------------------------------------------------------------------
# Cada no terminal mapea a una LISTA de producciones; cada producción es una
# lista de símbolos.  La cadena vacía se representa con EPSILON.

EPSILON = "ε"

GRAMMAR = {
    "E":  [["T", "E'"]],
    "E'": [["+", "T", "E'"], [EPSILON]],
    "T":  [["F", "T'"]],
    "T'": [["*", "F", "T'"], [EPSILON]],
    "F":  [["(", "E", ")"], ["id"]],
}

START_SYMBOL = "E"


# ---------------------------------------------------------------------------
# 2. Identificar terminales, no terminales y símbolo inicial
# ---------------------------------------------------------------------------
def analizar_gramatica(gramatica, inicio):
    """Devuelve (no_terminales, terminales) derivados de la gramática."""
    no_terminales = list(gramatica.keys())

    terminales = []
    for producciones in gramatica.values():
        for produccion in producciones:
            for simbolo in produccion:
                if simbolo == EPSILON:
                    continue
                if simbolo not in no_terminales and simbolo not in terminales:
                    terminales.append(simbolo)

    return no_terminales, terminales


def validar_gramatica(gramatica, inicio, no_terminales, terminales):
    """Comprueba que todo símbolo usado esté declarado. Lanza ValueError si no."""
    if inicio not in no_terminales:
        raise ValueError(f"El símbolo inicial '{inicio}' no es un no terminal.")
    for cabeza, producciones in gramatica.items():
        if not producciones:
            raise ValueError(f"El no terminal '{cabeza}' no tiene producciones.")
        for produccion in producciones:
            for simbolo in produccion:
                if simbolo != EPSILON and simbolo not in no_terminales and simbolo not in terminales:
                    raise ValueError(f"Símbolo desconocido '{simbolo}' en {cabeza}.")


# ---------------------------------------------------------------------------
# 3. FIRST de un símbolo y de una secuencia
# ---------------------------------------------------------------------------
def calcular_first(gramatica, no_terminales, terminales):
    """Cálculo iterativo (punto fijo) de FIRST para cada no terminal."""
    first = {nt: set() for nt in no_terminales}

    cambio = True
    while cambio:
        cambio = False
        for cabeza, producciones in gramatica.items():
            for produccion in producciones:
                primero = first_de_secuencia(produccion, first, no_terminales)
                antes = len(first[cabeza])
                first[cabeza] |= primero
                if len(first[cabeza]) != antes:
                    cambio = True

    return first


def first_de_secuencia(secuencia, first, no_terminales):
    """
    FIRST de una secuencia de símbolos.

    - Si el primer símbolo es un terminal, es ese terminal.
    - Si es un no terminal, se agrega su FIRST; si ese FIRST contiene ε, se
      continúa con el símbolo siguiente.
    - Si todos los símbolos pueden derivar ε, se agrega ε.
    """
    resultado = set()

    for simbolo in secuencia:
        if simbolo == EPSILON:
            resultado.add(EPSILON)
            return resultado

        if simbolo in no_terminales:
            resultado |= (first[simbolo] - {EPSILON})
            if EPSILON not in first[simbolo]:
                return resultado
        else:  # terminal
            resultado.add(simbolo)
            return resultado

    # Se recorrió toda la secuencia y todos podían derivar ε.
    resultado.add(EPSILON)
    return resultado


# ---------------------------------------------------------------------------
# 4. Cálculo iterativo de FOLLOW
# ---------------------------------------------------------------------------
def calcular_follow(gramatica, inicio, no_terminales, terminales, first):
    """Cálculo iterativo (punto fijo) de FOLLOW para cada no terminal."""
    follow = {nt: set() for nt in no_terminales}
    follow[inicio].add("$")          # marca de fin de entrada

    cambio = True
    while cambio:
        cambio = False
        for cabeza, producciones in gramatica.items():
            for produccion in producciones:
                for i, simbolo in enumerate(produccion):
                    if simbolo not in no_terminales:
                        continue

                    resto = produccion[i + 1:]
                    first_resto = first_de_secuencia(resto, first, no_terminales)

                    antes = len(follow[simbolo])
                    follow[simbolo] |= (first_resto - {EPSILON})

                    # Si el resto puede derivar ε (o no hay resto),
                    # FOLLOW(simbolo) incluye FOLLOW(cabeza).
                    if EPSILON in first_resto:
                        follow[simbolo] |= follow[cabeza]

                    if len(follow[simbolo]) != antes:
                        cambio = True

    return follow


# ---------------------------------------------------------------------------
# 5 y 6. Tabla predictiva LL(1) y detección de conflictos
# ---------------------------------------------------------------------------
def construir_tabla(gramatica, no_terminales, terminales, first, follow):
    """
    Tabla M[no_terminal][terminal] -> lista de producciones que la ocupan.

    Si una celda recibe más de una producción hay conflicto LL(1); se conserva
    la lista completa para poder informarlo.
    """
    tabla = {nt: {} for nt in no_terminales}

    for cabeza, producciones in gramatica.items():
        for produccion in producciones:
            first_prod = first_de_secuencia(produccion, first, no_terminales)

            for terminal in first_prod - {EPSILON}:
                tabla[cabeza].setdefault(terminal, []).append(produccion)

            if EPSILON in first_prod:
                for terminal in follow[cabeza]:
                    tabla[cabeza].setdefault(terminal, []).append(produccion)

    conflictos = []
    for nt in no_terminales:
        for terminal, producciones in tabla[nt].items():
            unicas = []
            for p in producciones:
                if p not in unicas:
                    unicas.append(p)
            if len(unicas) > 1:
                conflictos.append((nt, terminal, unicas))

    return tabla, conflictos


# ---------------------------------------------------------------------------
# 7. Impresión de resultados
# ---------------------------------------------------------------------------
def formatear_conjunto(conjunto, orden):
    """Ordena un conjunto según la lista `orden` para una salida estable."""
    conocidos = [s for s in orden if s in conjunto]
    desconocidos = sorted(s for s in conjunto if s not in orden)
    return "{ " + ", ".join(conocidos + desconocidos) + " }"


def formatear_produccion(cabeza, produccion):
    return f"{cabeza} -> {' '.join(produccion)}"


def imprimir_gramatica(gramatica, inicio, no_terminales, terminales):
    print("=" * 70)
    print("GRAMÁTICA DE TRABAJO")
    print("=" * 70)
    for cabeza, producciones in gramatica.items():
        alternativas = " | ".join(" ".join(p) for p in producciones)
        print(f"  {cabeza:<3} -> {alternativas}")

    print(f"\n  Símbolo inicial : {inicio}")
    print(f"  No terminales   : {', '.join(no_terminales)}")
    print(f"  Terminales      : {', '.join(terminales)}")
    print("  Fin de entrada  : $")
    print(f"  Cadena vacía    : {EPSILON}")


def imprimir_first(first, no_terminales, terminales):
    print("\n" + "=" * 70)
    print("CONJUNTOS FIRST")
    print("=" * 70)
    orden = terminales + [EPSILON]
    for nt in no_terminales:
        print(f"  FIRST({nt:<3}) = {formatear_conjunto(first[nt], orden)}")


def imprimir_follow(follow, no_terminales, terminales):
    print("\n" + "=" * 70)
    print("CONJUNTOS FOLLOW")
    print("=" * 70)
    orden = terminales + ["$"]
    for nt in no_terminales:
        print(f"  FOLLOW({nt:<3}) = {formatear_conjunto(follow[nt], orden)}")


def imprimir_tabla(tabla, no_terminales, terminales):
    print("\n" + "=" * 70)
    print("TABLA PREDICTIVA LL(1)   M[no terminal, terminal]")
    print("=" * 70)

    columnas = terminales + ["$"]
    ancho = max(14, *(len(c) + 2 for c in columnas))

    encabezado = " " * 6 + "".join(f"{c:^{ancho}}" for c in columnas)
    print(encabezado)
    print("-" * len(encabezado))

    for nt in no_terminales:
        fila = f"  {nt:<4}"
        for terminal in columnas:
            producciones = tabla[nt].get(terminal)
            if not producciones:
                celda = ""
            elif len(producciones) == 1:
                celda = formatear_produccion(nt, producciones[0])
            else:
                celda = "CONFLICTO"
            fila += f"{celda:^{ancho}}"
        print(fila)
    print()


def imprimir_conflictos(conflictos, grammar_es_ll1):
    print("=" * 70)
    print("DETECCIÓN DE CONFLICTOS LL(1)")
    print("=" * 70)
    if grammar_es_ll1:
        print("  No se encontraron conflictos: la gramática es LL(1).")
    else:
        print(f"  Se encontraron {len(conflictos)} celda(s) con conflicto:")
        for nt, terminal, producciones in conflictos:
            print(f"\n  Celda M[{nt}, {terminal}] recibe más de una producción:")
            for p in producciones:
                print(f"      - {formatear_produccion(nt, p)}")


# ---------------------------------------------------------------------------
# 8. Validación de cadenas con la tabla y una pila
# ---------------------------------------------------------------------------
def validar_cadena(cadena, tabla, inicio, terminales):
    """
    Análisis predictivo no recursivo con pila explícita.
    Devuelve (aceptada: bool, pasos: list[str], motivo: str).
    """
    tokens = cadena.split()
    entrada = tokens + ["$"]

    pila = ["$", inicio]
    pasos = []
    ip = 0  # índice del próximo token de entrada

    while True:
        tope = pila[-1]
        actual = entrada[ip]
        pila_txt = " ".join(reversed(pila))
        paso = f"{pila_txt:<28} | {cadena if cadena else '(vacía)':<22} | "

        if tope == "$" and actual == "$":
            pasos.append(paso + "ACEPTAR")
            return True, pasos, "Cadena aceptada."

        if tope == "$":
            pasos.append(paso + "ERROR")
            return False, pasos, (
                f"Error: la pila se vació pero queda entrada sin consumir "
                f"(token inesperado '{actual}')."
            )

        if tope in terminales:
            if tope == actual:
                pila.pop()
                ip += 1
                pasos.append(paso + f"emparejar '{tope}'")
            else:
                pasos.append(paso + "ERROR")
                return False, pasos, (
                    f"Error: se esperaba '{tope}' pero se encontró '{actual}' "
                    f"en la posición {ip + 1}."
                )
            continue

        # tope es un no terminal: consultar la tabla
        producciones = tabla[tope].get(actual)
        if not producciones:
            pasos.append(paso + "ERROR")
            return False, pasos, (
                f"Error: M[{tope}, {actual}] está vacía; no hay producción "
                f"aplicable en la posición {ip + 1}."
            )

        produccion = producciones[0]
        pila.pop()
        if produccion != [EPSILON]:
            for simbolo in reversed(produccion):
                pila.append(simbolo)
        pasos.append(paso + formatear_produccion(tope, produccion))


def imprimir_validaciones(tabla, inicio, terminales, validas, invalidas):
    print("\n" + "=" * 70)
    print("VALIDACIÓN DE CADENAS")
    print("=" * 70)

    def probar(cadena):
        aceptada, pasos, motivo = validar_cadena(cadena, tabla, inicio, terminales)
        print(f"\n  Cadena: {cadena}")
        print(f"  {'-' * (len(cadena) + 10)}")
        print(f"  {'PILA':<28} | {'ENTRADA':<22} | ACCIÓN")
        for paso in pasos:
            print(f"  {paso}")
        print(f"  Resultado: {'VÁLIDA' if aceptada else 'INVÁLIDA'} — {motivo}")
        return aceptada

    print("\n  --- Cadenas válidas (se espera que sean aceptadas) ---")
    resultados_validas = [probar(c) for c in validas]

    print("\n  --- Cadenas inválidas (se espera que sean rechazadas) ---")
    resultados_invalidas = [probar(c) for c in invalidas]

    print("\n" + "=" * 70)
    print("RESUMEN")
    print("=" * 70)

    ok_validas = all(resultados_validas)
    ok_invalidas = not any(resultados_invalidas)

    print("  Cadenas válidas:")
    for cadena, res in zip(validas, resultados_validas):
        print(f"    [{'OK' if res else 'FALLO'}] {cadena}")

    print("  Cadenas inválidas:")
    for cadena, res in zip(invalidas, resultados_invalidas):
        print(f"    [{'OK' if not res else 'FALLO'}] {cadena}")

    print(f"\n  Todas las válidas aceptadas   : {'SÍ' if ok_validas else 'NO'}")
    print(f"  Todas las inválidas rechazadas: {'SÍ' if ok_invalidas else 'NO'}")


# ---------------------------------------------------------------------------
# Programa principal
# ---------------------------------------------------------------------------
def main():
    no_terminales, terminales = analizar_gramatica(GRAMMAR, START_SYMBOL)
    validar_gramatica(GRAMMAR, START_SYMBOL, no_terminales, terminales)

    first = calcular_first(GRAMMAR, no_terminales, terminales)
    follow = calcular_follow(GRAMMAR, START_SYMBOL, no_terminales, terminales, first)
    tabla, conflictos = construir_tabla(
        GRAMMAR, no_terminales, terminales, first, follow
    )

    imprimir_gramatica(GRAMMAR, START_SYMBOL, no_terminales, terminales)
    imprimir_first(first, no_terminales, terminales)
    imprimir_follow(follow, no_terminales, terminales)
    imprimir_tabla(tabla, no_terminales, terminales)
    imprimir_conflictos(conflictos, len(conflictos) == 0)

    validas = ["id + id", "id * id", "id + id * id", "( id + id ) * id"]
    invalidas = ["id + * id", "id id", "( id + id", "id +"]
    imprimir_validaciones(tabla, START_SYMBOL, terminales, validas, invalidas)

    # Verificación automática de los conjuntos esperados
    print("\n" + "=" * 70)
    print("VERIFICACIÓN AUTOMÁTICA DE LOS CONJUNTOS")
    print("=" * 70)
    esperado_first = {
        "E":  {"(", "id"},
        "E'": {"+", EPSILON},
        "T":  {"(", "id"},
        "T'": {"*", EPSILON},
        "F":  {"(", "id"},
    }
    esperado_follow = {
        "E":  {")", "$"},
        "E'": {")", "$"},
        "T":  {"+", ")", "$"},
        "T'": {"+", ")", "$"},
        "F":  {"*", "+", ")", "$"},
    }
    for nt in no_terminales:
        ok_f = first[nt] == esperado_first[nt]
        ok_s = follow[nt] == esperado_follow[nt]
        print(f"  {nt:<3} FIRST {'OK' if ok_f else 'DISCREPANCIA'}   "
              f"FOLLOW {'OK' if ok_s else 'DISCREPANCIA'}")


if __name__ == "__main__":
    main()
